# Word2Vec: training your own model

In this notebook we will see how you can train your own Word2Vec model.

There are three main steps:
* Building a corpus from which to learn the embeddings.
* Preprocessing the corpus.
* Training the model.

We will be using `gensim` again, so we first need to install it:

In [ ]:
!pip install gensim

## 1. Building a corpus

As a toy example, we will train a Word2Vec model from a small corpus.

As an example, we will use a small Jane Austen corpus composed of the texts from two of her books: *Sense and Sensibility* and *Pride and Prejudice*.

Note that it is a VERY VERY small corpus! We would ideally train Word2Vec on larger datasets, to get meaningful embeddings.

First, download the files from Project Gutenberg ([Sense and Sensibility](https://www.gutenberg.org/cache/epub/161/pg161.txt) and [Pride and Prejudice](https://www.gutenberg.org/cache/epub/1342/pg1342.txt)) and save them in a new `data` folder, and manually remove the header and the end notice (or just run the following cell, which will do this for you!):

In [ ]:
!mkdir -p data
!wget -O data/sense.txt https://www.gutenberg.org/cache/epub/161/pg161.txt

with open("data/sense.txt") as fr:
    sensesensibility_text_lines = fr.readlines()
sensesensibility_text_lines = sensesensibility_text_lines[97:-349] # Remove header and end notice

!mkdir -p data
!wget -O data/pride.txt https://www.gutenberg.org/cache/epub/1342/pg1342.txt

with open("data/pride.txt") as fr:
    prideprejudice_text_lines = fr.readlines()
prideprejudice_text_lines = prideprejudice_text_lines[700:-366] # Remove header and end notice

In [ ]:
# The training set is the concatenation of the two texts (provided as lists of texts):
training_set = sensesensibility_text_lines + prideprejudice_text_lines

We have read the document with `.readlines()`, which returns a list of strings:

In [ ]:
# Print the type of `training_set`:
print(type(training_set))

# Print the length of `training_set`:
print(len(training_set))

# Print the type of the first element in `training_set`:
print(type(training_set[0]))

We now have a list of strings, each corresponding to one line in the books.

## 2. Preprocessing the corpus

### Preprocessing decisions

Many different choices can be made when preprocessing a corpus to be used as Word2Vec training data. Ultimately the choices you'll make will usually depend on:

1. The size of your corpus
2. What you are planning to use the model for

For this first complete example, our preprocessing will consist of:
- Lowercasing
- Punctuation removal

But preprocessing may also include tasks like:
- Stopword removal
- Lemmatisation
- Removal of numbers
- PoS tagging

The last few need a linguistic pipeline such as spaCy — see the optional notebook **4d-spaCy** if you want to add them.

### Reusing our own tokeniser

We do not need anything new here. In notebook **2g**, when we built the search engine, we wrote a small function that lowercased a text and threw away everything that was not a letter. That is exactly the preprocessing we need, so let's use it again.

In [ ]:
import re

The function below is the `tokenise` function from notebook 2g:

In [ ]:
def tokenize_text(sentence):
    """ turn a text into a list of comparable words

    Args:
        sentence: a string

    Returns:
        A list of lowercased words of at least three characters,
        with punctuation removed
    """
    sentence = sentence.lower()                    # lowercase
    sentence = sentence.strip()                    # strip white spaces
    sentence = re.sub(r"[^a-z' ]", " ", sentence)  # keep only letters and apostrophes
    tokens = sentence.split()                      # split on whitespace
    return tokens

⚠️ **What we lose by doing it this way.** spaCy would also give us lemmas, so that *walk*, *walks* and *walked* became one word, and it would let us drop stopwords by looking them up rather than by length. Our version treats every surface form as a separate word.

For word2vec that is a defensible choice — the model learns from how words are *used*, and inflected forms are used slightly differently — but it is a choice, and you should be able to say why you made it. If you want the linguistic version, notebook 4d shows how.

In [ ]:
# See how the function works on an example:
sentence = "These are test sentences, just to have a look at how a processed sentence looks like."

print(tokenize_text(sentence))

Applied to a whole corpus, this is what turns raw text into the list-of-lists that word2vec expects to be trained on.

In [ ]:
# word2vec trains on a list of tokenised documents — a list of lists:
example_corpus = [
    "It was the best of times, it was the worst of times.",
    "Call me Ishmael. Some years ago, never mind how long precisely.",
]

print([tokenize_text(text) for text in example_corpus])

### Preprocess your data

We will pre-process each text with our `tokenize_text` function.

In [ ]:
# Show the content of training_set:
print(training_set)

The output will be a list of lists, where each of the inner lists contains the relevant tokens in the text.

We can use the `tqdm` to track the progress of the processing (i.e. to show the progress bar).

In [ ]:
from tqdm import tqdm # To track the progress of the processing and training

In [ ]:
processed_data = [] # Empty list: that's where we will store our processed data.

# Iterate over all texts in our training set:
for text in tqdm(training_set):
    
    # Tokenise
    tokenized_text = tokenize_text(text)

    # Add the tokenized text (i.e. a list of tokens) to the list that will be
    # used to train the model.
    processed_data.append(tokenized_text)

In [ ]:
# Check the size of the `processed_data` variable
len(processed_data)

In [ ]:
# Get the first element of `processed_data` (i.e. the first text, tokenized and preprocessed):
processed_data[0]

Let's inspect the `processed_data` variable:

In [ ]:
# Print the type of processed_data:
print(type(processed_data))

# Print the length of processed_data:
print(len(processed_data))

# Print the type of the first element in the list:
print(type(processed_data[0]))

## 3. Training a word2vec model

The third step is to actually train the model.

We will use the Word2Vec module from Gensim to train a model.

In [ ]:
from gensim.models import Word2Vec

Before training, we have to make a few more decisions: we need to choose the **hyperparameters** we will use.

**Hyperparameters** are parameters whose values are decided by the user. Different choices of hyperparameters will have an impact on the resulting trained model.

[Here](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec) is a breakdown of all possible hyperparameters for training a model using Gensim's `Word2Vec`.

Hyperparameters are specified by the user when initialiasing a new Word2Vec model, before providing the data to learn from, and before starting the training.

If no hyperparameters are specified, the default ones will be used by the algorithm.

It's okay to start with the default ones, which correspond to the ones that were found by several studies to be optimal for many different tasks.

Normally, it is a good practice to find the **optimal parameters** for your model, depending on the planned use of your model.

You can instantiate Word2Vec (using the default parameters) as follows:

In [ ]:
# Instantiate Word2Vec with the default hyperparameters:
w2v_model = Word2Vec()

But you can also specify your choice of hyperparameters in the brackets:

In [ ]:
# Instantiate Word2Vec specifying some hyperparameters:
w2v_model = Word2Vec(min_count=1,             # how often a word should appear in order to be included
                     window=3,                # how many words before and after count as context
                     sg=1,                    # using the SkipGram algorithm (1) or the CBOW algorithm (0)?
                     vector_size=50,          # size of the vector
                    )

Before training, you will need to build the vocabulary. You can do it as follows:

In [ ]:
w2v_model.build_vocab(processed_data) # Build vocabulary

We then train the model as follows:

In [ ]:
w2v_model.train(processed_data, # tokenised data
                total_examples=len(processed_data), # Number of sentences to use for training
                epochs=30, #how many epochs to train for
                )

We have now trained the model! It was extremely fast because it is built from a super small corpus.

Let's check the type of the `w2v_model`:

In [ ]:
type(w2v_model)

We can extract the embeddings from the Word2Vec model with the `wv` attribute, as follows:

In [ ]:
embeddings = w2v_model.wv

Check the type of `embeddings`:

In [ ]:
type(embeddings)

Now that we have the KeyedVectors object, we can perform the vectors operations we showed in the previous notebook.

For example, we can check how many word embeddings have been learned:

In [ ]:
print(len(embeddings))

We will now save the vectors into a new file called `test-model-vectors.txt` in the `models/` directory.

The following cell uses the `pathlib` library to create a folder called `models` if such folder does not already exist.

In [ ]:
from pathlib import Path

Path('models/').mkdir(parents=True, exist_ok=True)

In [ ]:
embeddings.save_word2vec_format("models/test-model-vectors.txt", binary=False)

> Note: `binary=False` saves the vectors in a non-binary format (i.e. human-readable), which can take longer to store and process, but easier to deal with.

## Loading pre-trained embeddings

You can load the pre-trained embeddings (i.e. vectors) as shown in the previous notebook:

In [ ]:
# To load the full model, we need to import Word2Vec from gensim:
from gensim.models import KeyedVectors

In [ ]:
# To read a word2vec model, use the .load_word2vec_format() method, passing in the path to the model we just trained and saved:
our_test_vectors = KeyedVectors.load_word2vec_format('models/test-model-vectors.txt')

In [ ]:
# Check the data type:
type(our_test_vectors)

We can now access the embeddings as if in a dictionary:

In [ ]:
our_test_vectors["marianne"]

In [ ]:
# Get the most similar to word "marianne":
our_test_vectors.most_similar("marianne")

In [ ]:
# Get the most similar to word "london":
our_test_vectors.most_similar("london")

Well, maybe these semantic relations won't make a lot of sense. But this is because we've trained our embeddings using a very small (and diverse) corpus.

✏️ **Exercise:**

Obtain a corpus of text that is interesting for your research (medieval chronicles, the Shakespeare sonnets, Don Quijote, etc.).

Download it, preprocess it, train a Word2Vec model, explore the model.

In [ ]:
# Type your solution here:
